In [5]:
import os
import math
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from  sleio import sle_io
from utilies import *
from pathlib import Path
from subprocess import Popen, PIPE
from multiprocessing import Pool

# Phase 1 RUN RANDOM FIELDS GENERATION
# ────────────────────────────────────
# (20 processes)
# Total N fields (N*44*26*17)

#          ↓
# Phase 2  RUN SLE FORWARD SIMULATION
# ────────────────────────────────────
# (8 processes)
# SLE Forward Simulation

# Field 1 ~ Field N
# ├── inj_1
# ├── ...
# └── inj_8
#        ↓
# Total N fields (N*1*8*8)

# For HT-NN
# Input X dimension for HT-NN: 8x8 (inj_events * obs_num)
# Output Y dimension for HT-NN: 44x26x17 (core element size for sandbox )

## Conditional random field generation

* Prior information

In [8]:
# Prior K value using 16-th iteration from SLE inversion
iter = 16
file_name = "inverse/O-kestimate_prior.dat"
df_mean, df_var = parse_estimation(file_name, dim =3)

df_prior = df_mean[['x', 'y', 'z', f'iter_{iter}']].copy()
df_prior['lnK_Var'] = df_var[f'iter_{iter}'].values
df_prior['K'] = df_prior[f'iter_{iter}'].values
df_prior['log_K'] = np.log(df_prior[f'iter_{iter}'].values)

df_prior_core = df_prior[
    (df_prior['x'] >= 132) & (df_prior['x'] <= 445) &
    (df_prior['y'] >= 96) & (df_prior['y'] <= 286) &
    (df_prior['z'] >= 95) & (df_prior['z'] <= 195)
]

* geostatistics analysis (no need if done already)

In [9]:
xg, yg, zg, inter_inK = interpolate_3d(df_prior_core, value_col=f'iter_{iter}', grid_size=[120, 60, 80], method="linear")

In [3]:
df_geo['x']

NameError: name 'df_geo' is not defined

In [2]:
df_geo = pd.DataFrame({'x': xg.flatten(),
                       'y': yg.flatten(),
                       'z': zg.flatten(),
                       'mean': inter_inK.flatten()})
# plot_3d_isosurfaces(df_geo, 'ln_mean', 5, [-10, -7], 'SLE lnK interpolation')

bins_x = np.arange(0,  200,  10)
bins_y = np.arange(0,  150,  10)
bins_z = np.arange(0,  120,  10)

plt.figure(figsize = (10, 6))
variogram_test(coordinate = (xg,yg,zg), value=df_geo['mean'], bins= bins_x, direction='X', max_dis = 200)
plt.show()
variogram_test(coordinate = (xg,yg,zg), value=df_geo['mean'], bins= bins_y, direction='Y', max_dis = 150)
plt.show()
variogram_test(coordinate = (xg,yg,zg), value=df_geo['mean'], bins= bins_z, direction='Z', max_dis = 110)
plt.show()

NameError: name 'pd' is not defined

* crf parameters

In [ ]:
injA_data  = [
        ("injA_1",  288.5,  101, 150),
        ("injA_2",  288.5,  146, 150),
        ("injA_3",  143.5,  191, 150),
        ("injA_4",  196,	191, 150),
        ("injA_5",  241,	191, 150),
        ("injA_6",  336,	191, 150),
        ("injA_7",  381,	191, 150),
        ("injA_8",  433.5,  191, 150),
        ("injA_9",  288.5,  236, 150),
        ("injA_10", 288.5,  281, 150),
        # ("Middle", 288.5,  191, 150,)
    ]

injA_df = pd.DataFrame(injA_data, columns=["name", "x", "y", "z"])
# plt.scatter(injA_df['x'], injA_df['y'])

#-- Setting dataframe for conditional random field
df_est = pd.DataFrame({'x': df_prior['x'],
                       'y': df_prior['y'],
                       'z': df_prior['z'],
                       'K': df_prior['K'],
                       'ln_var': df_prior['lnK_Var']})

coordinate = df_est[['x', 'y', 'z']]

#-- prior geostatistics and crf condition info
crf_params = {
    "ens_num":4,
    "seed": 20260629,
    "model": "Gaussian",
    "variance": 0.09,
    "correlation_scale": {"x": 50, "y": 60, "z": 47},
    "c_scale_mul": 1.5,
    "inj_A_num": len(injA_df),
    "section_sample_num": 1,
    "fraction_rate": 0.25,
    "resample_time": 2,
    "contain_rate": 0.5
    }

for key, para in crf_params.items():
    print(f'{key}: {para}')

ens_num: 4
seed: 20260629
model: Gaussian
variance: 0.09
correlation_scale: {'x': 50, 'y': 60, 'z': 47}
c_scale_mul: 1.5
inj_A_num: 10
section_sample_num: 1
fraction_rate: 0.25
resample_time: 2
contain_rate: 0.5


* generate random fields

In [ ]:
crf_list = []
rng = MasterRNG(crf_params["seed"])
df_est['weight'] = d2_weight(df_est, injA_df, 10**6)

df_est_core = df_est[
    (df_est['x'] >= 132) & (df_est['x'] <= 445) &
    (df_est['y'] >= 96) &  (df_est['y'] <= 286) &
    (df_est['z'] >= 95) &  (df_est['z'] <= 195) ].copy()

for i in range(crf_params['resample_time']):
    
    sub_num = crf_params['ens_num']//crf_params['resample_time']
    print(f'process resample group {i+1}/{crf_params["resample_time"]}')
    print("-"*30)
    
    df_con = sampler(df_est, crf_params)
    # print(len(df_con))
    
    for j in range(sub_num):
        seed = int(rng())
        process_index_ =  sub_num*i + j
        crf = crf_generator(coordinate, df_con, crf_params, seed, j+1) 
        # plt.scatter(df_est['K'].values, crf)
        print(f"Generating field {process_index_+1}   lnK mean:{np.mean(crf):.3f}")    
        
        
        crf_list.append(crf)
        
        df_crf = pd.DataFrame({
            'x': coordinate['x'].values,
            'y': coordinate['y'].values,
            'z': coordinate['z'].values,
            'value': crf,
        })
        
        df_crf_core = df_crf[
            (df_crf['x'] >= 132) & (df_crf['x'] <= 445) &
            (df_crf['y'] >= 96) & (df_crf['y'] <= 286) &
            (df_est['z'] >= 95) &  (df_est['z'] <= 195)
        ]
        print(f"{ r2_score(df_est_core['K'].values, df_crf_core['value']):.5f}")
        fig = plot_crf_isosurfaces(df_crf, 'value', df_con, 8, [1, 2], f'Crf_{j+1}')
        # fig = plot_3d_isosurfaces(df_crf_core, 'value', 8, [1, 2], 'K')
        fig.show()

## Forward framework

In [6]:
# Create sandbox forward simulation
# Each injection will be a independent events
# Each event contain 1 source well, 8 observations well
# All events keep the same observation well (ob_1~ ob_8)

# Simulation control (unit: mm, sec)
project_name = 'Sandbox_forward'
simulation_control = ('3D', 'steady', 'confined')  # dimension, problem, aquifer

# Geometry control
dx = np.array( [43.5, 25, 20, 20, 20] + [7.5] * 19 + [6, 6, 5.5, 5.5, 6, 6] + [7.5] * 19 + [20, 20, 20, 25, 43.5])
dy = np.array( [31, 20, 20, 20, 10] + [7.5] * 24 + [10, 20, 20, 20, 31])
dz = np.array( [30, 25, 20, 15, 10, 10] + [5] * 11 + [7.5, 7.5] + [10, 10, 10, 25, 30, 35] )

start_coord = (0, 0, 0)         # origin coordinate (x0, y0, z0)
element_num = (54, 34, 25)      # element number in each direction (e_x, e_y, e_z)
element_spacing = (dx, dy, dz)  # element spacing in each direction (len(dx) = e_x-)

# initial condition
init_paras = (350, 0, 0.004, 0.00001, 0.4) # init_h, init_flux, init_K, init_Ss, porosity

# boundary condition
boundary = ('surface', 'UP', 'head')  # method, value, bc_type, bc_value

# injection well info (Each inj_N is a event)
const_rate = 600
inj_time = (0, 600)
inj_1 = ((288.5, 71,  120), 'inj_1', {'rate':const_rate, 'time': inj_time})
inj_2 = ((88.5,  71,  110), 'inj_2', {'rate':const_rate, 'time': inj_time})
inj_3 = ((88.5,  191, 100), 'inj_3', {'rate':const_rate, 'time': inj_time})
inj_4 = ((88.5,  311, 90 ), 'inj_4', {'rate':const_rate, 'time': inj_time})
inj_5 = ((288.5, 311, 120), 'inj_5', {'rate':const_rate, 'time': inj_time})
inj_6 = ((488.5, 311, 110), 'inj_6', {'rate':const_rate, 'time': inj_time})
inj_7 = ((488.5, 191, 100), 'inj_7', {'rate':const_rate, 'time': inj_time})
inj_8 = ((488.5, 71,  90 ), 'inj_8', {'rate':const_rate, 'time': inj_time})

injs = [inj_1, inj_2, inj_3,inj_4, inj_5, inj_6, inj_7, inj_8]

# observation well info (same for all events)
ob_1 = ((188.5, 71,  130), 'Pie_1')
ob_2 = ((88.5, 131,  200), 'Pie_2')
ob_3 = ((88.5, 251,  180), 'Pie_3')
ob_4 = ((188.5, 311, 150), 'Pie_4')
ob_5 = ((388.5, 311, 150), 'Pie_5')
ob_6 = ((488.5, 251, 180), 'Pie_6')
ob_7 = ((488.5, 131, 200), 'Pie_7')
ob_8 = ((388.5, 71,  130), 'Pie_8')

obs = [ob_1, ob_2, ob_3, ob_4, ob_5, ob_6, ob_7, ob_8]

# time info for transient (Not used for steady case)
flag = 'each_time_step'
dt, dt_max, dt_mul, t_max, max_red = (1, 2, 1, 60, 2)
time = (dt, dt_max, dt_mul, t_max, max_red, flag)      
                                

In [ ]:
# project/
# │
# ├── material/
# │   ├── material_1.dat
# │   ├── material_2.dat
# │   ├── ...
# │   └── material_5.dat
# │
# ├── forward/
# │   │
# │   ├── Event_1/
# │   │   ├── SLE.exe
# │   │   ├── grid.dat
# │   │   ├── node.dat
# │   │   ├── element.dat
# │   │   ├── problem.dat
# │   │   └── ...
# │   │
# │   ├── Event_2/
# │   │   ├── SLE.exe
# │   │   ├── grid.dat
# │   │   └── ...
# │   │
# │   ├── ...
# │   │
# │   └── Event_8/
# │       ├── SLE.exe
# │       ├── grid.dat
# │       └── ...

stress_num = len(injs)
event_name = [f"inj_{i+1}" for i in range(stress_num )]

for i in range(stress_num):
    
    project_name = f"forward_{event_name[i]}"
    folder = Path(os.getcwd()) / "forward" / event_name[i]
    fw_ = sle_io(project_name, folder)

    # Geometry 
    fw_.set_parameters(simulation_control)
    fw_.create_geometry(start_coord, element_num, element_spacing)
    fw_.create_initial(init_paras)
    fw_.create_boundary(boundary)

    # events
    fw_.add_stress(1, time, obs, (injs[i],))
    fw_.create_times()
    fw_.create_source()
    fw_.create_observation()

    # # simulation 
    fw_.create_function()
    fw_.create_simulation()
    fw_.create_problem()
    fw_.write_forward_input()
    
    print("-"*36)